<!-- paper-paired-role-banner -->
> 🟩 **정답 코드 · 오른쪽 창** — 왼쪽과 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 구현 이유와 텐서 shape를 설명해 보세요.

# 14. DQN — Playing Atari with Deep Reinforcement Learning

- **원 논문:** [Playing Atari with Deep Reinforcement Learning](https://arxiv.org/abs/1312.5602)
- **저자 / 발표:** Volodymyr Mnih et al. · arXiv preprint (2013)
- **난이도 / 예상 시간:** 중급 · 약 65분
- **선수 지식:** PyTorch 기본, 강화학습의 state/action/reward, Bellman equation

이 노트북은 논문 결과 수치를 그대로 재현하는 대규모 benchmark가 아니라,
핵심 수식과 구조가 실제로 어떻게 동작하는지 확인하는 **하드웨어 인식 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하며, 작은 NumPy·Python 실험은
전송 비용을 피하기 위해 CPU에서 실행될 수 있습니다.

## 재현 범위와 완료 기준

**이 노트북이 재현하는 것**

외부 Gym 없이 5-state LineWorld에서 replay memory, epsilon-greedy 행동, 고정된 Bellman target과 Q-network MSE update를 구현해 greedy 정책이 오른쪽 보상에 도달하게 한다.

**원 논문과 다른 것**

논문은 Atari 화면 4장을 convolutional Q-network에 넣어 수백만 frame을 학습한다. 여기서는 one-hot 상태 5개와 2층 MLP로 Algorithm 1의 학습 역학만 재현한다.

완료하려면 TODO를 구현하고, 검증 셀의 `assert`를 모두 통과시키고, 마지막 관찰 질문에
자신의 말로 답하세요. 수치가 논문과 다르더라도 핵심 불변식과 비교 방향이 맞으면 성공입니다.

## 논문 ↔ 노트북 지도

        절 번호와 수식 번호는 위 링크의 대표 공개본을 기준으로 합니다. 판본에 따라 페이지는 달라질 수
        있으므로 **절 제목과 수식 번호**를 함께 사용하세요.

        | 원 논문의 위치 | 이 노트북의 대응 실습 | 확인할 증거 |
        |---|---|---|
        | §2 Background, Eq. (1) | `bellman_targets`의 terminal-aware TD target | done 상태에는 bootstrap 값이 더해지지 않는지 검사 |
| §2, Eq. (2)–(3) | `QNetwork`가 선택한 action의 Q와 고정 target 사이 오차를 최소화 | Q-value shape, finite loss와 gradient를 검사 |
| §4 Deep Reinforcement Learning, Algorithm 1 | `ReplayBuffer`와 `epsilon_greedy`의 수집·표본화·탐험 경로 | buffer batch 계약과 학습 뒤 greedy return을 검사 |

        > 읽기 순서: 먼저 해당 절을 훑고 → 코드를 예측해 작성하고 → 출력이 논문의 주장과 왜
        > 연결되는지 한 문장으로 설명합니다.

## 핵심 수식과 구현 설계

$$
y=r+\gamma(1-d)\max_{a'}Q_{\theta^-}(s',a'),\qquad
\mathcal{L}(\theta)=\mathbb{E}[(y-Q_\theta(s,a))^2]
$$

- $\theta$는 online network, $\theta^-$는 고정된 target network, $d$는 terminal flag입니다.
- replay buffer는 상관된 transition을 섞고 target network는 bootstrapping target을 느리게 변화시킵니다.
- transition 저장, batch sampling, TD target, optimization, target sync를 명시적으로 나눕니다.
- state batch $[B,D_s]$에서 action value $[B,A]$를 만들고 gather로 $Q(s,a):[B]$를 선택합니다.

## 구현 선택 이유

- **원문에서 보존한 부분:** §2 Background, Eq. (1)
- **노트북 구현:** `bellman_targets`의 terminal-aware TD target
- **이렇게 구현한 이유:** 완성된 고수준 model을 한 줄로 호출하면
  논문의 핵심 shape·mask·loss·상태 변화가 숨겨집니다. 따라서 이 계산을
  이름이 드러나는 class와 함수로 나누고 중간 tensor를 assertion으로
  검사합니다.
- **실패를 잡는 증거:** done 상태에는 bootstrap 값이 더해지지 않는지 검사를 출력 크기만 아니라
  구현 불변식으로 사용합니다.
- **축소 선택:** 논문은 Atari 화면 4장을 convolutional Q-network에 넣어 수백만 frame을 학습한다. 여기서는 one-hot 상태 5개와 2층 MLP로 Algorithm 1의 학습 역학만 재현한다.는 기본 실행에서 생략합니다.
  이는 Windows CPU에서도 반복 실험이 가능하게 하면서 핵심 메커니즘은
  유지하기 위한 선택입니다.

## 1. Atari DQN을 LineWorld로 축소하기

논문 §2 Eq. (1)의 Bellman 관계에서 한 step target은

$$y=r+\gamma(1-\text{done})\max_{a'}Q(s',a')$$

입니다. §2 Eq. (2)는 $(y-Q(s,a;\theta))^2$을 최소화하고, §4 Algorithm 1은
상관된 연속 경험 대신 replay memory에서 무작위 전이를 뽑습니다. 여기서는 상태 0과
4가 terminal인 다섯 칸 환경을 사용합니다. 4에 도착하면 +1, 0이면 -1입니다.

**성공 기준:** 학습 뒤 greedy agent의 평균 return이 0.8보다 크고, 시작 상태에서
`Q(right) > Q(left)`여야 합니다.

### 구현 단계 1 · 실행 환경·데이터 계약 (setup 예외)

- **원문 위치:** 해당 없음 — import, seed, device, 로컬 데이터 로딩만 담당하며 논문의 학습 연산을 구현하지 않습니다.
- **Tensor shape 계약:** states [B,Dₛ] → action values [B,A] → selected Q/TD target [B]
- **구현 이유:** 이후 셀의 비교가 재현 가능하도록 장치·dtype·seed와 입력 불변식을 먼저 고정합니다.
- **완료 증거:** 데이터 존재 여부, 입력 rank, 장치 선택 assertion과 환경 요약 출력이 통과해야 합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.manual_seed,np.random.seed,np.clip,torch.as_tensor,F.one_hot -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 5개 · 수식 2개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.manual_seed(seed)</code></summary>

#### `torch.manual_seed(seed)`

- **역할:** PyTorch 난수 생성기의 시드를 설정합니다.
- **입력·반환:** 정수 시드를 받고 `Generator`를 반환합니다.
- **이 셀에서 쓰는 이유:** 가중치 초기화와 텐서 샘플링을 반복 가능하게 만듭니다.
- **주의점:** 완전한 결정성에는 알고리즘·장치 설정도 추가로 필요할 수 있습니다.
- **공식 문서:** [torch.manual_seed](https://docs.pytorch.org/docs/stable/generated/torch.manual_seed.html)

</details>

<details>
<summary><code>np.random.seed(seed=None)</code></summary>

#### `np.random.seed(seed=None)`

- **역할:** NumPy의 기존 전역 난수 생성기 상태를 초기화합니다.
- **입력·반환:** 시드를 받고 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** 레거시 `np.random.*` 호출의 수열을 재현합니다.
- **주의점:** 새 코드에서는 독립 상태를 가진 `default_rng`가 권장됩니다.
- **공식 문서:** [np.random.seed](https://numpy.org/doc/stable/reference/random/generated/numpy.random.seed.html)

</details>

<details>
<summary><code>np.clip(a, a_min=None, a_max=None)</code></summary>

#### `np.clip(a, a_min=None, a_max=None)`

- **역할:** 배열 값을 범위 안으로 제한합니다.
- **입력·반환:** 배열과 축·dtype 등의 설정을 받고 배열 또는 수치 결과를 반환합니다.
- **이 셀에서 쓰는 이유:** 데이터 shape과 수치 연산을 논문 실험에 맞게 구성·검증합니다.
- **주의점:** 축, dtype, 복사·view 여부와 빈 배열에서의 동작을 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
y=\min\!\left(\max(x,\ell),u\right)
$$

- **기호:** $\ell$은 하한, $u$는 상한입니다.
- **수식 → 코드:** 코드의 `min`·`max` 바깥 값은 각각 경계값으로 치환됩니다.
- **직관:** 수치적으로 위험한 범위를 잘라내지만 경계 밖의 차이는 모두 잃습니다.
- **shape 확인:** 입력과 출력 shape은 같습니다.
- **공식 문서:** [np.clip](https://numpy.org/doc/stable/reference/generated/numpy.clip.html)

</details>

<details>
<summary><code>torch.as_tensor(data, dtype=None, device=None)</code></summary>

#### `torch.as_tensor(data, dtype=None, device=None)`

- **역할:** 가능하면 데이터를 공유해 텐서로 해석합니다.
- **입력·반환:** 배열형 데이터를 받고 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 불필요한 복사를 줄이며 입력을 텐서 계약으로 맞춥니다.
- **주의점:** 원본과 저장공간을 공유할 수 있어 변경 부작용을 확인해야 합니다.
- **공식 문서:** [torch.as_tensor](https://docs.pytorch.org/docs/stable/generated/torch.as_tensor.html)

</details>

<details>
<summary><code>F.one_hot(tensor, num_classes=-1)</code></summary>

#### `F.one_hot(tensor, num_classes=-1)`

- **역할:** 정수 class ID를 one-hot 텐서로 변환합니다.
- **입력·반환:** 예측·target과 설정을 받고 축약된 loss 또는 같은 shape의 loss를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 objective를 수치화해 역전파 시작점으로 사용합니다.
- **주의점:** 입력 형식, target dtype·shape, `reduction` 의미를 반드시 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
H_{i,c}=\mathbb{1}[x_i=c]
$$

- **기호:** $x_i$는 class ID이고 $H_{i,c}$는 ID가 $c$일 때만 1인 indicator입니다.
- **수식 → 코드:** API가 정수 index를 class 축이 추가된 indicator tensor로 바꿉니다.
- **직관:** 범주 번호를 거리 있는 수로 보지 않고 서로 독립적인 축으로 표현합니다.
- **shape 확인:** 입력 `[...] -> [...,C]`; 출력은 정수 dtype이므로 loss에 따라 float 변환이 필요합니다.
- **공식 문서:** [F.one_hot](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.one_hot.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
from collections import deque
import math
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from llm_engineering_lab.acceleration import get_accelerator

torch.manual_seed(140)
np.random.seed(140)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
# Python environment steps stay on CPU; AI_LAB_DEVICE=cpu can avoid
# accelerator overhead for this very small neural update.


class LineWorld:
    n_states = 5
    n_actions = 2  # 0=left, 1=right

    def reset(self):
        self.state = 2
        self.steps = 0
        return self.state

    def step(self, action):
        self.steps += 1
        self.state = int(np.clip(self.state + (-1 if action == 0 else 1), 0, 4))
        done = self.state in (0, 4) or self.steps >= 8
        reward = 1.0 if self.state == 4 else (-1.0 if self.state == 0 else -0.02)
        return self.state, reward, done


def encode_states(states):
    states = torch.as_tensor(states, dtype=torch.long)
    return F.one_hot(states, num_classes=LineWorld.n_states).float().to(DEVICE)


env = LineWorld()
print(ACCELERATOR.summary())
print("LineWorld and replay storage stay on CPU; neural batches use DEVICE.")
assert env.reset() == 2
assert encode_states([0, 2, 4]).shape == (3, 5)

### 구현 단계 2 · 핵심 연산·모델

- **원문 위치:** §4 Deep Reinforcement Learning, Algorithm 1
- **이 셀의 책임:** `ReplayBuffer`와 `epsilon_greedy`의 수집·표본화·탐험 경로
- **Tensor shape 계약:** states [B,Dₛ] → action values [B,A] → selected Q/TD target [B]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** buffer batch 계약과 학습 뒤 greedy return을 검사. 코드의 assertion과 출력으로 바로 확인합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.tensor,torch.float32,np.random.default_rng -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 3개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.tensor(data, dtype=None, device=None)</code></summary>

#### `torch.tensor(data, dtype=None, device=None)`

- **역할:** 데이터를 복사해 새 PyTorch 텐서를 만듭니다.
- **입력·반환:** 배열형 데이터와 선택적 dtype·device를 받고 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 상수·label·축소 실험 입력을 명시적으로 구성합니다.
- **주의점:** 기존 텐서에서 호출하면 복사가 생기므로 경우에 따라 `as_tensor`가 낫습니다.
- **공식 문서:** [torch.tensor](https://docs.pytorch.org/docs/stable/generated/torch.tensor.html)

</details>

<details>
<summary><code>torch.float32</code></summary>

#### `torch.float32`

- **역할:** 32비트 부동소수점 dtype
- **입력·반환:** 값이 아니라 텐서 원소의 저장 형식으로 사용됩니다.
- **이 셀에서 쓰는 이유:** 일반적인 학습 실수 dtype입니다.
- **주의점:** 연산마다 허용 dtype이 다르고 변환 시 정밀도나 메모리가 달라집니다.
- **공식 문서:** [torch.float32](https://docs.pytorch.org/docs/stable/tensor_attributes.html#torch.dtype)

</details>

<details>
<summary><code>np.random.default_rng(seed=None)</code></summary>

#### `np.random.default_rng(seed=None)`

- **역할:** 독립적인 NumPy 난수 생성기 `Generator`를 만듭니다.
- **입력·반환:** 선택적 시드를 받고 `Generator` 객체를 반환합니다.
- **이 셀에서 쓰는 이유:** 전역 상태를 오염시키지 않고 데이터나 실험 난수를 재현합니다.
- **주의점:** 생성기를 다시 만들지 않으면 호출할 때마다 내부 상태가 진행됩니다.
- **공식 문서:** [np.random.default_rng](https://numpy.org/doc/stable/reference/random/generator.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
class ReplayBuffer:
    def __init__(self, capacity):
        self.capacity = capacity
        self.storage = []
        self.position = 0

    def __len__(self):
        return len(self.storage)

    def push(self, state, action, reward, next_state, done):
        transition = (state, action, reward, next_state, done)
        if len(self.storage) < self.capacity:
            self.storage.append(transition)
        else:
            self.storage[self.position] = transition
        self.position = (self.position + 1) % self.capacity

    def sample(self, batch_size, rng):
        indices = rng.choice(len(self.storage), size=batch_size, replace=False)
        batch = [self.storage[int(index)] for index in indices]
        states, actions, rewards, next_states, dones = zip(*batch)
        return (
            torch.tensor(states, dtype=torch.long),
            torch.tensor(actions, dtype=torch.long),
            torch.tensor(rewards, dtype=torch.float32),
            torch.tensor(next_states, dtype=torch.long),
            torch.tensor(dones, dtype=torch.float32),
        )


probe_buffer = ReplayBuffer(3)
for i in range(5):
    probe_buffer.push(i % 4, i % 2, float(i), (i + 1) % 5, False)
assert len(probe_buffer) == 3
probe_batch = probe_buffer.sample(2, np.random.default_rng(141))
assert all(tensor.shape == (2,) for tensor in probe_batch)

### 구현 단계 3 · 핵심 연산·모델

- **원문 위치:** §2 Background, Eq. (1) / §2, Eq. (2)–(3)
- **이 셀의 책임:** `bellman_targets`의 terminal-aware TD target / `QNetwork`가 선택한 action의 Q와 고정 target 사이 오차를 최소화
- **Tensor shape 계약:** states [B,Dₛ] → action values [B,A] → selected Q/TD target [B]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** done 상태에는 bootstrap 값이 더해지지 않는지 검사 / Q-value shape, finite loss와 gradient를 검사. 코드의 assertion과 출력으로 바로 확인합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=nn.Sequential,nn.Linear,nn.ReLU,Tensor.detach,Tensor.requires_grad_,Module.eval,Module.parameters,torch.sqrt,torch.allclose -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 9개 · 수식 4개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>nn.Sequential(*modules)</code></summary>

#### `nn.Sequential(*modules)`

- **역할:** 모듈을 순서대로 연결합니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.
- **공식 문서:** [nn.Sequential](https://docs.pytorch.org/docs/stable/generated/torch.nn.Sequential.html)

</details>

<details>
<summary><code>nn.Linear(in_features, out_features, bias=True)</code></summary>

#### `nn.Linear(in_features, out_features, bias=True)`

- **역할:** 완전연결 affine 변환을 만듭니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
y=xW^{\mathsf T}+b
$$

- **기호:** $x$는 입력, $W$는 weight, $b$는 bias, $y$는 출력입니다.
- **수식 → 코드:** `in_features`와 `out_features`가 각각 $W$의 두 축을 결정합니다.
- **직관:** 각 출력 feature는 모든 입력 feature의 학습 가능한 가중합입니다.
- **shape 확인:** `[..., D_in] -> [..., D_out]`, weight는 `[D_out, D_in]`입니다.
- **공식 문서:** [nn.Linear](https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html)

</details>

<details>
<summary><code>nn.ReLU(inplace=False)</code></summary>

#### `nn.ReLU(inplace=False)`

- **역할:** 음수 값을 0으로 만드는 활성함수를 만듭니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
\operatorname{ReLU}(x)=\max(0,x)
$$

- **기호:** $x$는 선형 변환 뒤 activation입니다.
- **수식 → 코드:** 코드가 음수 원소를 0으로 치환하고 양수는 그대로 통과시킵니다.
- **직관:** 단순하지만 여러 선형 층을 비선형 함수로 조합할 수 있게 합니다.
- **shape 확인:** 입력과 출력 shape은 같습니다.
- **공식 문서:** [nn.ReLU](https://docs.pytorch.org/docs/stable/generated/torch.nn.ReLU.html)

</details>

<details>
<summary><code>tensor.detach()</code></summary>

#### `tensor.detach()`

- **역할:** 같은 저장공간을 공유하면서 현재 계산 그래프에서 분리한 텐서를 만듭니다.
- **입력·반환:** 텐서를 받아 gradient를 추적하지 않는 텐서 view를 반환합니다.
- **이 셀에서 쓰는 이유:** 로그·teacher target·NumPy 변환이 역전파 경로를 만들지 않게 합니다.
- **주의점:** 원본과 저장공간을 공유하므로 in-place 변경은 양쪽에 영향을 줄 수 있습니다.
- **공식 문서:** [Tensor.detach](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.detach.html)

</details>

<details>
<summary><code>tensor.requires_grad_(requires_grad=True)</code></summary>

#### `tensor.requires_grad_(requires_grad=True)`

- **역할:** 텐서의 gradient 추적 여부를 in-place로 바꿉니다.
- **입력·반환:** 불리언을 받고 같은 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** teacher 동결이나 새로 학습할 파라미터 범위를 명시합니다.
- **주의점:** 이미 만들어진 계산 그래프를 소급해 바꾸지 않으며 leaf 여부를 확인해야 합니다.
- **공식 문서:** [Tensor.requires_grad_](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.requires_grad_.html)

</details>

<details>
<summary><code>model.eval()</code></summary>

#### `model.eval()`

- **역할:** 모듈과 하위 모듈을 평가 모드로 전환합니다.
- **입력·반환:** 인자 없이 호출하고 모듈 자신을 반환합니다.
- **이 셀에서 쓰는 이유:** Dropout을 끄고 BatchNorm의 저장 통계를 사용하게 합니다.
- **주의점:** gradient 기록은 끄지 않으므로 필요하면 `no_grad`도 사용합니다.
- **공식 문서:** [Module.eval](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html)

</details>

<details>
<summary><code>model.parameters(recurse=True)</code></summary>

#### `model.parameters(recurse=True)`

- **역할:** 모듈에 등록된 학습 파라미터 iterator를 반환합니다.
- **입력·반환:** 하위 모듈 포함 여부를 받고 `Parameter` iterator를 반환합니다.
- **이 셀에서 쓰는 이유:** optimizer가 갱신할 weight와 bias를 전달합니다.
- **주의점:** iterator는 한 번 소비되며 buffer와 일반 tensor attribute는 포함하지 않습니다.
- **공식 문서:** [Module.parameters](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html)

</details>

<details>
<summary><code>torch.sqrt(input)</code></summary>

#### `torch.sqrt(input)`

- **역할:** 각 원소의 제곱근을 계산합니다.
- **입력·반환:** 0 이상 텐서를 받고 같은 shape의 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 표준편차, 거리, diffusion schedule의 scale을 계산합니다.
- **주의점:** 음수 입력은 NaN을 만들며 0 근처 gradient가 매우 커질 수 있습니다.

##### 관련 수식과 코드 연결

$$
y_i=\sqrt{x_i},\qquad x_i\ge 0
$$

- **기호:** $x_i$는 음수가 아닌 입력입니다.
- **수식 → 코드:** API가 제곱 또는 분산 scale을 원래 단위의 길이로 되돌립니다.
- **직관:** 제곱된 크기를 사람이 해석하기 쉬운 원래 scale로 바꿉니다.
- **shape 확인:** 입력과 출력 shape은 같습니다.
- **공식 문서:** [torch.sqrt](https://docs.pytorch.org/docs/stable/generated/torch.sqrt.html)

</details>

<details>
<summary><code>torch.allclose(input, other, rtol=1e-5, atol=1e-8, equal_nan=False)</code></summary>

#### `torch.allclose(input, other, rtol=1e-5, atol=1e-8, equal_nan=False)`

- **역할:** 두 텐서의 모든 원소가 허용 오차 안에서 가까운지 검사합니다.
- **입력·반환:** 두 텐서와 오차 설정을 받고 Python `bool` 하나를 반환합니다.
- **이 셀에서 쓰는 이유:** 장치나 구현을 바꾼 뒤 수치 결과가 사실상 같은지 확인합니다.
- **주의점:** 정확한 동일성 검사가 아니며 작은 값에서는 `atol`을 직접 정해야 합니다.

##### 관련 수식과 코드 연결

$$
\lvert a_i-b_i\rvert \le \mathrm{atol}+\mathrm{rtol}\,\lvert b_i\rvert
$$

- **기호:** $a_i$는 계산값, $b_i$는 기준값, `atol`은 절대 오차, `rtol`은 상대 오차 허용치입니다.
- **수식 → 코드:** `isclose`는 이 부등식을 원소마다 계산하고, `allclose`는 모든 결과에 논리 AND를 적용합니다.
- **직관:** 완전히 같은 비트인지가 아니라 실험 목적에 충분히 가까운지를 묻는 검사입니다.
- **shape 확인:** 두 입력은 broadcasting 가능해야 합니다. `isclose`는 broadcast shape, `allclose`는 scalar `bool`을 반환합니다.
- **공식 문서:** [torch.allclose](https://docs.pytorch.org/docs/stable/generated/torch.allclose.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
class QNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(LineWorld.n_states, 24),
            nn.ReLU(),
            nn.Linear(24, LineWorld.n_actions),
        )

    def forward(self, encoded_state):
        return self.net(encoded_state)


def bellman_targets(rewards, dones, next_q_values, gamma=0.95):
    # Eq. (2)의 theta_(i-1) 역할: target을 gradient graph에서 분리한다.
    best_next = next_q_values.max(dim=1).values.detach()
    return rewards + gamma * (1.0 - dones) * best_next


def hard_sync_target(online_network, target_network):
    target_network.load_state_dict(online_network.state_dict())
    target_network.requires_grad_(False)
    target_network.eval()


def parameter_distance(first_network, second_network):
    squared_distance = 0.0
    parameter_pairs = zip(
        first_network.parameters(),
        second_network.parameters(),
        strict=True,
    )
    for first_parameter, second_parameter in parameter_pairs:
        difference = first_parameter.detach() - second_parameter.detach()
        squared_distance = squared_distance + difference.square().sum()
    return float(torch.sqrt(squared_distance).cpu())


q_network = QNetwork().to(DEVICE)
target_network = QNetwork().to(DEVICE)
hard_sync_target(q_network, target_network)
probe_next_q = torch.tensor([[2.0, 3.0], [8.0, 9.0]], requires_grad=True)
probe_target = bellman_targets(
    torch.tensor([1.0, -1.0]),
    torch.tensor([0.0, 1.0]),
    probe_next_q,
)
assert torch.allclose(probe_target, torch.tensor([3.85, -1.0]))
assert not probe_target.requires_grad
assert q_network(encode_states([0, 1])).shape == (2, 2)
assert q_network is not target_network
assert all(not parameter.requires_grad for parameter in target_network.parameters())
assert parameter_distance(q_network, target_network) == 0.0

### 구현 단계 4 · 목적함수·학습 update

- **원문 위치:** §2 Background, Eq. (1) / §2, Eq. (2)–(3) / §4 Deep Reinforcement Learning, Algorithm 1
- **이 셀의 책임:** `bellman_targets`의 terminal-aware TD target / `QNetwork`가 선택한 action의 Q와 고정 target 사이 오차를 최소화 / `ReplayBuffer`와 `epsilon_greedy`의 수집·표본화·탐험 경로
- **Tensor shape 계약:** states [B,Dₛ] → action values [B,A] → selected Q/TD target [B]
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** done 상태에는 bootstrap 값이 더해지지 않는지 검사 / Q-value shape, finite loss와 gradient를 검사 / buffer batch 계약과 학습 뒤 greedy return을 검사. 코드의 assertion과 출력으로 바로 확인합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.no_grad,torch.optim.Adam,Optimizer.step,ACCELERATOR.move,F.mse_loss,Optimizer.zero_grad,Tensor.backward,np.isfinite -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 8개 · 수식 4개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.no_grad()</code></summary>

#### `torch.no_grad()`

- **역할:** 블록 안에서 autograd 기록을 끄는 context manager입니다.
- **입력·반환:** 보통 `with`에 사용하며 블록의 텐서는 기본적으로 gradient를 기록하지 않습니다.
- **이 셀에서 쓰는 이유:** 평가나 고정 teacher 추론의 메모리와 연산 부담을 줄입니다.
- **주의점:** `model.eval()`과 역할이 다르므로 평가 모드가 필요하면 둘 다 사용합니다.
- **공식 문서:** [torch.no_grad](https://docs.pytorch.org/docs/stable/generated/torch.no_grad.html)

</details>

<details>
<summary><code>torch.optim.Adam(params, lr=1e-3, ...)</code></summary>

#### `torch.optim.Adam(params, lr=1e-3, ...)`

- **역할:** 1·2차 gradient 모멘트를 추적하는 Adam optimizer를 만듭니다.
- **입력·반환:** 파라미터와 학습률 등을 받고 optimizer 객체를 반환합니다.
- **이 셀에서 쓰는 이유:** 노트북의 trainable parameter를 반복 update합니다.
- **주의점:** weight decay의 해석은 AdamW와 다르며 학습률이 결과에 크게 작용합니다.

##### 관련 수식과 코드 연결

$$
\begin{aligned}m_t&=\beta_1m_{t-1}+(1-\beta_1)g_t,\\v_t&=\beta_2v_{t-1}+(1-\beta_2)g_t^2,\\\theta_t&=\theta_{t-1}-\eta\frac{\hat m_t}{\sqrt{\hat v_t}+\epsilon}\end{aligned}
$$

- **기호:** $g_t$는 gradient, $m_t,v_t$는 1·2차 모멘트, $\eta$는 학습률입니다.
- **수식 → 코드:** `backward()`가 만든 gradient를 읽어 모멘트를 갱신한 뒤 `step()`에서 파라미터에 적용합니다.
- **직관:** 최근 gradient 방향과 크기를 함께 누적해 파라미터별 적응형 보폭을 만듭니다.
- **shape 확인:** 각 trainable parameter와 같은 shape의 모멘트 상태가 optimizer에 저장됩니다.
- **공식 문서:** [torch.optim.Adam](https://docs.pytorch.org/docs/stable/generated/torch.optim.Adam.html)

</details>

<details>
<summary><code>optimizer.step(closure=None)</code></summary>

#### `optimizer.step(closure=None)`

- **역할:** 현재 gradient와 optimizer 규칙으로 파라미터를 갱신합니다.
- **입력·반환:** 선택적 closure를 받고 optimizer에 따라 loss 또는 `None`을 반환합니다.
- **이 셀에서 쓰는 이유:** 역전파로 계산한 gradient를 실제 학습 update에 반영합니다.
- **주의점:** 일반적으로 `zero_grad`, forward, `backward` 다음에 호출합니다.

##### 관련 수식과 코드 연결

$$
\theta_{t+1}=\mathcal{U}(\theta_t,\nabla_\theta L_t,s_t;\eta)
$$

- **기호:** $\mathcal U$는 optimizer update, $s_t$는 momentum·moment 상태입니다.
- **수식 → 코드:** `step()`은 이미 `.grad`에 저장된 값을 읽어 선택한 optimizer 규칙을 적용합니다.
- **직관:** gradient 계산과 parameter 갱신을 분리하면 학습 lifecycle을 정확히 추적할 수 있습니다.
- **shape 확인:** parameter shape은 유지되고 값과 optimizer state만 바뀝니다.
- **공식 문서:** [Optimizer.step](https://docs.pytorch.org/docs/stable/generated/torch.optim.Optimizer.step.html)

</details>

<details>
<summary><code>ACCELERATOR.move(value)</code></summary>

#### `ACCELERATOR.move(value)`

- **역할:** 텐서나 모듈을 선택된 실행 장치로 옮기는 프로젝트 도우미입니다.
- **입력·반환:** 이동할 값을 받고 같은 논리 값을 장치에 배치해 반환합니다.
- **이 셀에서 쓰는 이유:** 모델과 batch가 서로 다른 장치에 놓이는 오류를 막습니다.
- **주의점:** 컨테이너 재귀 처리 여부와 dtype 변경 여부는 프로젝트 구현을 확인해야 합니다.
- **공식 문서:** 프로젝트 내부 도우미이므로 정의의 docstring을 확인합니다.

</details>

<details>
<summary><code>F.mse_loss(input, target, reduction=&#x27;mean&#x27;)</code></summary>

#### `F.mse_loss(input, target, reduction='mean')`

- **역할:** 예측과 target의 제곱 오차를 계산합니다.
- **입력·반환:** 예측·target과 설정을 받고 축약된 loss 또는 같은 shape의 loss를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 objective를 수치화해 역전파 시작점으로 사용합니다.
- **주의점:** 입력 형식, target dtype·shape, `reduction` 의미를 반드시 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
\mathcal{L}_{MSE}=\frac{1}{N}\sum_{i=1}^{N}(\hat y_i-y_i)^2
$$

- **기호:** $\hat y_i$는 예측, $y_i$는 target입니다.
- **수식 → 코드:** `reduction='mean'`이 모든 원소의 제곱 오차를 평균냅니다.
- **직관:** 큰 오차를 제곱해 더 강하게 벌점하므로 회귀와 reconstruction에 자주 씁니다.
- **shape 확인:** 예측과 target은 같거나 broadcast 가능한 shape이어야 합니다.
- **공식 문서:** [F.mse_loss](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.mse_loss.html)

</details>

<details>
<summary><code>optimizer.zero_grad(set_to_none=True)</code></summary>

#### `optimizer.zero_grad(set_to_none=True)`

- **역할:** 파라미터에 누적된 gradient를 비웁니다.
- **입력·반환:** 초기화 방식을 받고 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** 현재 mini-batch의 gradient만으로 update하도록 학습 단계를 시작합니다.
- **주의점:** 호출하지 않으면 PyTorch gradient가 기본적으로 누적됩니다.
- **공식 문서:** [Optimizer.zero_grad](https://docs.pytorch.org/docs/stable/generated/torch.optim.Optimizer.zero_grad.html)

</details>

<details>
<summary><code>loss.backward()</code></summary>

#### `loss.backward()`

- **역할:** 스칼라 loss에서 계산 그래프를 거슬러 gradient를 계산합니다.
- **입력·반환:** 스칼라 텐서에서는 보통 인자 없이 호출하며 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** optimizer update에 필요한 각 파라미터의 `.grad`를 채웁니다.
- **주의점:** 같은 그래프를 다시 역전파하려면 별도 보존 설정이 필요합니다.

##### 관련 수식과 코드 연결

$$
\frac{\partial L}{\partial x}=\frac{\partial L}{\partial y}\frac{\partial y}{\partial x},\qquad x.\mathrm{grad}\leftarrow x.\mathrm{grad}+\nabla_xL
$$

- **기호:** $L$은 scalar loss이고 $x$는 계산 그래프의 leaf parameter입니다.
- **수식 → 코드:** `backward()`는 연쇄법칙으로 gradient를 계산해 각 parameter의 `.grad`에 누적합니다.
- **직관:** 파라미터를 직접 바꾸는 호출이 아니라 update에 필요한 미분값을 준비하는 호출입니다.
- **shape 확인:** 각 `.grad`는 대응 parameter와 같은 shape이며 scalar가 아닌 출력에는 seed gradient가 필요합니다.
- **공식 문서:** [Tensor.backward](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.backward.html)

</details>

<details>
<summary><code>np.isfinite(x)</code></summary>

#### `np.isfinite(x)`

- **역할:** 각 값이 NaN이나 무한대가 아닌 유한수인지 검사합니다.
- **입력·반환:** 배열형 입력을 받고 같은 shape의 불리언 배열을 반환합니다.
- **이 셀에서 쓰는 이유:** 학습 값이나 전처리 결과가 수치적으로 정상인지 검증합니다.
- **주의점:** 유한하다는 사실만 확인하며 값의 크기가 적절하다는 뜻은 아닙니다.
- **공식 문서:** [np.isfinite](https://numpy.org/doc/stable/reference/generated/numpy.isfinite.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
def epsilon_greedy(network, state, epsilon, rng):
    if rng.random() < epsilon:
        return int(rng.integers(LineWorld.n_actions))
    with torch.no_grad():
        return int(network(encode_states([state])).argmax(dim=1).item())


replay = ReplayBuffer(capacity=400)
optimizer = torch.optim.Adam(q_network.parameters(), lr=8e-3)
rng = np.random.default_rng(142)
returns, losses = [], []
sync_interval = 40
update_count = 0
pre_sync_distances = []
post_sync_distances = []

for episode in range(180):
    state = env.reset()
    episode_return = 0.0
    epsilon = max(0.05, 0.85 * (1.0 - episode / 150.0))
    done = False
    while not done:
        action = epsilon_greedy(q_network, state, epsilon, rng)
        next_state, reward, done = env.step(action)
        replay.push(state, action, reward, next_state, done)
        state = next_state
        episode_return += reward

        if len(replay) >= 32:
            states, actions, rewards, next_states, dones = replay.sample(32, rng)
            states, actions, rewards, next_states, dones = ACCELERATOR.move(
                states, actions, rewards, next_states, dones
            )
            predicted_q = q_network(encode_states(states))
            predicted_q = predicted_q.gather(
                1,
                actions[:, None],
            ).squeeze(1)
            with torch.no_grad():
                next_q = target_network(encode_states(next_states))
                targets = bellman_targets(rewards, dones, next_q)
            loss = F.mse_loss(predicted_q, targets)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            losses.append(float(loss.detach()))
            update_count += 1

            if update_count % sync_interval == 0:
                distance_before = parameter_distance(
                    q_network,
                    target_network,
                )
                pre_sync_distances.append(distance_before)
                assert distance_before > 0.0
                hard_sync_target(q_network, target_network)
                distance_after = parameter_distance(
                    q_network,
                    target_network,
                )
                post_sync_distances.append(distance_after)
                assert distance_after == 0.0

    returns.append(episode_return)

print(f"episodes={len(returns)}, replay={len(replay)}, updates={len(losses)}")
assert len(returns) == 180 and len(losses) > 100
assert np.isfinite(losses).all()
assert pre_sync_distances and min(pre_sync_distances) > 0.0
assert post_sync_distances and max(post_sync_distances) == 0.0
assert all(parameter.grad is None for parameter in target_network.parameters())

### 구현 단계 5 · 평가·완료 증거

- **원문 위치:** §4 Deep Reinforcement Learning, Algorithm 1
- **이 셀의 책임:** `ReplayBuffer`와 `epsilon_greedy`의 수집·표본화·탐험 경로
- **Tensor shape 계약:** states [B,Dₛ] → action values [B,A] → selected Q/TD target [B]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** buffer batch 계약과 학습 뒤 greedy return을 검사. 코드의 assertion과 출력으로 바로 확인합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.arange,np.mean -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 2개 · 수식 1개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.arange(start=0, end, step=1, ...)</code></summary>

#### `torch.arange(start=0, end, step=1, ...)`

- **역할:** 일정 간격의 1차원 텐서를 만듭니다.
- **입력·반환:** 범위와 간격을 받고 1차원 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** position·index·time step을 벡터화해 만듭니다.
- **주의점:** 부동소수점 step은 반올림 때문에 원소 수가 예상과 다를 수 있습니다.
- **공식 문서:** [torch.arange](https://docs.pytorch.org/docs/stable/generated/torch.arange.html)

</details>

<details>
<summary><code>np.mean(a, axis=None, dtype=None, ...)</code></summary>

#### `np.mean(a, axis=None, dtype=None, ...)`

- **역할:** 지정 축의 산술평균을 계산합니다.
- **입력·반환:** 배열과 축·dtype 등의 설정을 받고 배열 또는 수치 결과를 반환합니다.
- **이 셀에서 쓰는 이유:** 데이터 shape과 수치 연산을 논문 실험에 맞게 구성·검증합니다.
- **주의점:** 축, dtype, 복사·view 여부와 빈 배열에서의 동작을 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
\bar x=\frac{1}{N}\sum_{i=1}^{N}x_i
$$

- **기호:** $N$은 선택한 축의 원소 수입니다.
- **수식 → 코드:** `dim` 또는 `axis`가 평균에 포함할 원소와 결과에서 사라질 축을 결정합니다.
- **직관:** 여러 관측값을 중심 위치 하나로 요약합니다.
- **shape 확인:** `keepdim=False`이면 축약한 축이 제거되고 참이면 크기 1로 남습니다.
- **공식 문서:** [np.mean](https://numpy.org/doc/stable/reference/generated/numpy.mean.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
evaluation_returns = []
eval_rng = np.random.default_rng(143)
for _ in range(40):
    state = env.reset()
    total = 0.0
    done = False
    while not done:
        action = epsilon_greedy(q_network, state, 0.0, eval_rng)
        state, reward, done = env.step(action)
        total += reward
    evaluation_returns.append(total)

with torch.no_grad():
    all_states = encode_states(torch.arange(LineWorld.n_states))
    all_q = q_network(all_states).detach().cpu().numpy()
mean_return = float(np.mean(evaluation_returns))
print(f"greedy mean return={mean_return:.3f}")
print("Q(start=2) [left, right] =", np.round(all_q[2], 3).tolist())
assert mean_return > 0.80
assert all_q[2, 1] > all_q[2, 0]

moving_return = np.convolve(returns, np.ones(15) / 15, mode="valid")
fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))
axes[0].plot(returns, alpha=0.3, label="episode return")
axes[0].plot(np.arange(14, len(returns)), moving_return, label="15-episode mean")
axes[0].set(title="epsilon-greedy learning", xlabel="episode")
axes[0].legend()
axes[1].plot(losses)
axes[1].set(title="replay minibatch MSE", xlabel="update")
states_axis = np.arange(LineWorld.n_states)
axes[2].plot(states_axis, all_q[:, 0], "o-", label="left")
axes[2].plot(states_axis, all_q[:, 1], "o-", label="right")
axes[2].set(title="learned action values", xlabel="state", ylabel="Q")
axes[2].legend()
fig.tight_layout()
plt.show()

## 7. 해석 예시 답안

1. 시간적으로 강하게 상관된 표본을 섞어 update 분산을 낮추고, 한 경험을 여러 번 재사용해 데이터 효율을 높입니다.
2. Eq. (2)는 이전/고정 Q가 만든 target을 상수로 보고 현재 Q만 회귀합니다.
   detach하지 않으면 target 쪽 Q도 동시에 움직여 다른 목적함수가 됩니다.
3. 원 논문은 최근 4개 Atari frame을 convolution으로 표현하지만, 여기서는
   Markov 상태를 이미 알고 one-hot 벡터로 제공합니다. replay와 Bellman
   update는 같지만 시각표현 학습은 생략했습니다.

## 마무리 해설과 다음 실험

1. 핵심 연산과 구현 위치를 원 논문의 용어로 다시 연결해 설명하세요.
2. 이 미니 실험은 `논문은 Atari 화면 4장을 convolutional Q-network에 넣어 수백만 frame을 학습한다. 여기서는 one-hot 상태 5개와 2층 MLP로 Algorithm 1의 학습 역학만 재현한다.`를 재현하지 않으므로 논문의 최종 성능을 검증한 것이 아닙니다.
3. 다음에는 데이터 크기, 모델 폭, 학습 횟수 중 **하나만** 바꾸고 같은 seed에서 metric을 비교하세요.

정답의 수치 자체보다, shape·확률 정규화·mask·gradient 같은 불변식을 먼저 확인하는 습관이
논문 구현에서 가장 강한 디버깅 도구입니다.